# Simulación de Panel: Programa de Crédito Productivo

## Características del modelo

- **Panel de datos**: N empresas × T períodos
- **Inicio del programa configurable**: Período $t_0$
- **Cohortes secuenciales con cupos**
- **Evolución temporal realista** de outcomes
- **Efecto demostración** entre cohortes
- **Múltiples outcomes**: empleados (entero), salario (positivo), acceso a crédito (binario)

### Modelo de efectos dinámicos

$$\tau_{it}(k) = \tau^{imm} + \tau^{grad} \cdot \min(k, k_{max})$$

Donde $k$ = períodos desde el tratamiento.

---
## CONFIGURACIÓN COMPLETA DEL PROGRAMA
---

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from config import CONFIG

print(f"Configuración: {CONFIG['nombre_programa']}")
print(f"  Empresas: {CONFIG['n_empresas']:,}")
print(f"  Períodos: {CONFIG['n_periodos']} ({CONFIG['frecuencia']})")
print(f"  Inicio programa: Período {CONFIG['periodo_inicio_programa']}")
print(f"  Cohortes: {CONFIG['n_cohortes']} con cupos {CONFIG['cupo_por_cohorte']}")

---
## IMPLEMENTACIÓN DEL SIMULADOR
---

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import warnings

from scipy.special import expit
from typing import Dict, List, Set

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

---
## EJECUCIÓN
---

In [ ]:
from PanelCreditSimulator import PanelCreditSimulator

simulator = PanelCreditSimulator(CONFIG)
panel = simulator.simulate()

In [ ]:
panel

In [ ]:
# Resumen
ever_treated = panel.groupby('firm_id')['tratado'].max()
n_treated = ever_treated.sum()
eligible_ever = panel.groupby('firm_id')['elegible'].max()
n_control = (eligible_ever & ~ever_treated).sum()

print(f"\nESTRUCTURA DEL PANEL")
print("=" * 70)
print(f"Dimensiones: {panel.shape[0]:,} obs ({CONFIG['n_empresas']:,} × {CONFIG['n_periodos']} períodos)")
print(f"Empresas tratadas: {n_treated}")
print(f"Empresas control: {n_control}")
print(f"\nPeríodo programa: {CONFIG['periodo_inicio_programa']} ({panel[panel['periodo']==CONFIG['periodo_inicio_programa']]['fecha'].iloc[0]})")

In [ ]:
# Verificación de tipos de datos
print("\nVERIFICACIÓN DE TIPOS")
print("=" * 70)
print(f"Empleados:")
print(f"  • Tipo: {panel['empleados'].dtype}")
print(f"  • Rango: [{panel['empleados'].min()}, {panel['empleados'].max()}]")
print(f"  • ¿Enteros?: {(panel['empleados'] == panel['empleados'].astype(int)).all()}")

print(f"\nSalario promedio:")
print(f"  • Rango: [${panel['salario_promedio'].min():,.0f}, ${panel['salario_promedio'].max():,.0f}]")
print(f"  • ¿Todos positivos?: {(panel['salario_promedio'] > 0).all()}")

print(f"\nTiene crédito:")
print(f"  • Valores únicos: {sorted(panel['tiene_credito'].unique())}")

In [ ]:
# Vista del panel
print("\nVISTA DEL PANEL (empresa tratada)")
print("=" * 70)
treated_firms = list(ever_treated[ever_treated].index)
sample = panel[panel['firm_id'] == treated_firms[0]]
display(sample[['firm_id', 'fecha', 'empleados', 'salario_promedio', 'tiene_credito', 'tratado', 'cohort']].round(0))

---
## VISUALIZACIÓN
---

In [ ]:
def plot_evolution(panel: pd.DataFrame, config: Dict):
    """Visualiza evolución temporal."""
    t0 = config['periodo_inicio_programa']
    
    ever_treated = panel.groupby('firm_id')['tratado'].max()
    treated_firms = set(ever_treated[ever_treated].index)
    eligible_ever = panel.groupby('firm_id')['elegible'].max()
    control_firms = set(eligible_ever[eligible_ever].index) - treated_firms
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # 1. Empleados
    ax = axes[0, 0]
    for firms, color, label in [(treated_firms, '#2ecc71', 'Tratadas'), 
                                 (control_firms, '#3498db', 'Control')]:
        means = panel[panel['firm_id'].isin(firms)].groupby('periodo')['empleados'].mean()
        ax.plot(means.index, means.values, color=color, lw=2, label=label, marker='o', ms=4)
    ax.axvline(t0, color='red', ls='--', alpha=0.7, label='Inicio')
    ax.set_xlabel('Período')
    ax.set_ylabel('Empleados')
    ax.set_title('Evolución de Empleados')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. Salario
    ax = axes[0, 1]
    for firms, color, label in [(treated_firms, '#2ecc71', 'Tratadas'), 
                                 (control_firms, '#3498db', 'Control')]:
        means = panel[panel['firm_id'].isin(firms)].groupby('periodo')['salario_promedio'].mean()
        ax.plot(means.index, means.values/1000, color=color, lw=2, label=label, marker='o', ms=4)
    ax.axvline(t0, color='red', ls='--', alpha=0.7)
    ax.set_xlabel('Período')
    ax.set_ylabel('Salario (miles)')
    ax.set_title('Evolución de Salario')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. Crédito
    ax = axes[0, 2]
    for firms, color, label in [(treated_firms, '#2ecc71', 'Tratadas'), 
                                 (control_firms, '#3498db', 'Control')]:
        means = panel[panel['firm_id'].isin(firms)].groupby('periodo')['tiene_credito'].mean()
        ax.plot(means.index, means.values*100, color=color, lw=2, label=label, marker='o', ms=4)
    ax.axvline(t0, color='red', ls='--', alpha=0.7)
    ax.set_xlabel('Período')
    ax.set_ylabel('% con crédito')
    ax.set_title('Acceso a Crédito')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 4. Diferencia T-C (empleados)
    ax = axes[1, 0]
    t_means = panel[panel['firm_id'].isin(treated_firms)].groupby('periodo')['empleados'].mean()
    c_means = panel[panel['firm_id'].isin(control_firms)].groupby('periodo')['empleados'].mean()
    diff = t_means - c_means
    colors = ['#3498db' if t < t0 else '#e74c3c' for t in diff.index]
    ax.bar(diff.index, diff.values, color=colors, alpha=0.7)
    ax.axvline(t0-0.5, color='black', ls='--', lw=2)
    ax.axhline(diff[diff.index < t0].mean(), color='#3498db', ls=':', lw=2)
    ax.axhline(diff[diff.index >= t0].mean(), color='#e74c3c', ls=':', lw=2)
    ax.set_xlabel('Período')
    ax.set_ylabel('Diferencia T-C')
    ax.set_title('DiD: Empleados')
    ax.grid(True, alpha=0.3)
    
    # 5. Por cohorte
    ax = axes[1, 1]
    n_coh = config['n_cohortes']
    colors_coh = plt.cm.viridis(np.linspace(0.2, 0.8, n_coh))
    for c in range(n_coh):
        coh_firms = panel[panel['cohort'] == c]['firm_id'].unique()
        if len(coh_firms) > 0:
            means = panel[panel['firm_id'].isin(coh_firms)].groupby('periodo')['empleados'].mean()
            ax.plot(means.index, means.values, color=colors_coh[c], lw=2, label=f'Cohorte {c}', marker='o', ms=3)
    c_means = panel[panel['firm_id'].isin(control_firms)].groupby('periodo')['empleados'].mean()
    ax.plot(c_means.index, c_means.values, color='gray', lw=2, ls='--', label='Control')
    ax.axvline(t0, color='red', ls='--', alpha=0.5)
    ax.set_xlabel('Período')
    ax.set_ylabel('Empleados')
    ax.set_title('Por Cohorte')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # 6. Event study
    ax = axes[1, 2]
    treated_panel = panel[panel['firm_id'].isin(treated_firms)].copy()
    first_treat = panel[panel['tratado']].groupby('firm_id')['periodo'].min()
    treated_panel['rel_period'] = treated_panel.apply(
        lambda x: x['periodo'] - first_treat.get(x['firm_id'], 0), axis=1
    )
    event_means = treated_panel.groupby('rel_period')['empleados'].mean()
    event_means = event_means[(event_means.index >= -5) & (event_means.index <= 7)]
    if -1 in event_means.index:
        event_means = event_means - event_means[-1]
    ax.bar(event_means.index, event_means.values, color='steelblue', alpha=0.7)
    ax.axhline(0, color='black', lw=1)
    ax.axvline(-0.5, color='red', ls='--')
    ax.set_xlabel('Período relativo')
    ax.set_ylabel('Δ Empleados')
    ax.set_title('Event Study')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_evolution(panel, CONFIG)

---
## ESTIMACIONES DiD
---

In [ ]:
def compute_did(panel: pd.DataFrame, config: Dict):
    """Calcula DiD simple."""
    t0 = config['periodo_inicio_programa']

    ever_treated = panel.groupby('firm_id')['tratado'].max()
    treated_firms = set(ever_treated[ever_treated].index)
    eligible_ever = panel.groupby('firm_id')['elegible'].max()
    control_firms = set(eligible_ever[eligible_ever].index) - treated_firms

    pre = panel[panel['periodo'] < t0]
    post = panel[panel['periodo'] >= t0]

    results = []
    for outcome in ['empleados', 'salario_promedio', 'tiene_credito']:
        pre_t = pre[pre['firm_id'].isin(treated_firms)][outcome].mean()
        pre_c = pre[pre['firm_id'].isin(control_firms)][outcome].mean()
        post_t = post[post['firm_id'].isin(treated_firms)][outcome].mean()
        post_c = post[post['firm_id'].isin(control_firms)][outcome].mean()

        did = (post_t - pre_t) - (post_c - pre_c)

        results.append({
            'Outcome': outcome,
            'Pre_T': pre_t,
            'Pre_C': pre_c,
            'Post_T': post_t,
            'Post_C': post_c,
            'DiD': did
        })

    return pd.DataFrame(results)

did_df = compute_did(panel, CONFIG)
print("\nESTIMACIONES DiD")
print("=" * 70)
display(did_df.round(2))

---
## EXPORTACIÓN
---

In [ ]:
def export_panel(panel: pd.DataFrame, path: str, exclude_unobs: bool = True):
    """Exporta panel."""
    if exclude_unobs:
        unobs = ['calidad_gerencial', 'productividad_latente', 'propension_credito']
        cols = [c for c in panel.columns if c not in unobs]
        export_df = panel[cols]
    else:
        export_df = panel

    export_df.to_csv(path, index=False)
    print(f"Exportado: {path} ({export_df.shape})")

# Ejemplo:
export_panel(panel, 'panel_credito.csv')